# Toy ANN2SNN BrainScaleS-2 hardware-in-the-loop

Use the `EBRAINS-experimental` kernel. This notebook only configures the hardware client, invokes the repository CLI, and displays artifacts; experiment logic remains in `brainscales2_toy_hil.py`.

The default resume run reuses the accepted seed-0 checkpoint and explicit calibrations, then executes TTFS corrected-max pooling. Set `RESUME_SOURCE_RUN = None` for a fresh train-and-probe run. Hagen initialization is isolated behind a fixed watchdog before expensive stages begin.

In [ ]:
%pip install --quiet --disable-pip-version-check jaxtyping matplotlib

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import csv
import json
import math
import os
import signal
import subprocess
import sys
import time

start = Path.cwd().resolve()
repo_root = next((p for p in (start, *start.parents) if (p / 'scripts/evaluation/brainscales2_toy_hil.py').is_file()), None)
if repo_root is None:
    raise RuntimeError('Could not locate the delayed-temporal repository root')
os.chdir(repo_root)
CLI = repo_root / 'scripts/evaluation/brainscales2_toy_hil.py'

# By default, resume from the accepted seed-0 checkpoint and calibrations.
# Set RESUME_SOURCE_RUN = None to train and probe a fresh checkpoint.
RESUME_SOURCE_RUN = repo_root / 'artifacts/brainscales2-toy/20260901T095827Z'
RUN_SERVICE_PREFLIGHT = True
RUN_TRAIN = RESUME_SOURCE_RUN is None
RUN_LOCAL_REPLAY = False
RUN_HAGEN_PROBE = RESUME_SOURCE_RUN is None
RUN_MARGIN_CALIBRATION = True
RUN_HARDWARE_SMOKE = True
RUN_YINYANG_FULL = True
RUN_MNIST_BENCHMARK = False

CHECKPOINT_DIR = (RESUME_SOURCE_RUN / 'checkpoint' if RESUME_SOURCE_RUN else None)
ARTIFACT_ROOT = None  # Set an existing run directory to resume completed conditions.
HAGEN_CALIBRATION_PATH = (
    RESUME_SOURCE_RUN / 'calibration/hagen_cocolist.pbin'
    if RESUME_SOURCE_RUN else None
)
SPIKING_CALIBRATION_PATH = (
    RESUME_SOURCE_RUN / 'calibration/spiking_cocolist.pbin'
    if RESUME_SOURCE_RUN else None
)
DEADLINE_MARGIN_PATH = None  # Existing deadline_margin.json when resuming.
HAGEN_HIDDEN_SHIFT = 2 if RESUME_SOURCE_RUN else 1
TOY_ACTIVATION = 'relu'  # Train a separate 'sigmoid' checkpoint for the bounded-activation control.
RELU_BOUNDARY = 'implicit-lower-bound-host'  # Used only for TOY_ACTIVATION='relu'.
POOLING_DOMAIN = 'ttfs'  # Physical LIF replicas; required for corrected-max pooling.
TEMPORAL_POOL_ESTIMATOR = 'analytic-corrected-max'
BASE_DEADLINE_S = 60.0e-6
MARGIN_DIAGNOSTIC_DEADLINE_S = 100.0e-6
MARGIN_MAX_S = 40.0e-6
MARGIN_STEP_S = 1.0e-6
MARGIN_TARGET_SAMPLE_MISS_RATE = 0.05
MARGIN_CONFIDENCE = 0.95
MARGIN_CALIBRATION_SAMPLES = 256
MARGIN_CALIBRATION_TRIALS = 8
MARGIN_BOOTSTRAP_ITERATIONS = 2000
SPIKING_THRESHOLD = 125
SPIKING_INPUT_FAN_IN = 4  # Accepted primitive operating point.
NEURON_WEIGHT_CALIBRATION_PATH = None  # Validated file; use its matching fan-in.
POOL_SAMPLE_CHUNK_SIZE = 64  # Bounds full-run hxtorch memory.
POOL_REPLICA_SAMPLE_BUDGET = 128  # 2 GB session: M=8 uses 16; M=16 uses 8.
POOL_CALIBRATION_TRIAL_CHUNK_SIZE = 4  # 2 GB session: 32 codes x 4 trials per worker.
HAGEN_ROW_CHUNK_SIZE = 128  # Bounds physical first and output PWM batches.
CONDITION_WORKER_MAX_ATTEMPTS = 3  # Retries transient EBRAINS RPC failures.
CONDITION_WORKER_RETRY_BACKOFF_S = 20.0
CONDITION_WORKER_IDLE_TIMEOUT_S = 180.0  # Kill a worker after 3 min without output.
MNIST_HARDWARE_SAMPLES = 128  # Inspect runtime.json before increasing this.
SMOKE_MAX_MULTI_SPIKE_RATE = 0.05
HARDWARE_CLIENT_MAX_ATTEMPTS = 3
HARDWARE_CLIENT_RETRY_BACKOFF_S = 5.0
PREFLIGHT_TIMEOUT_S = 120
CLI_TIMEOUTS_S = {
    'train': 60 * 60,
    'convert': 15 * 60,
    'local-eval': 60 * 60,
    'probe-hagen': 20 * 60,
    'calibrate-margin': 2 * 60 * 60,
    'hardware-smoke': 2 * 60 * 60,
    'hardware-eval': 12 * 60 * 60,
}

run_label = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
artifact_root = (
    Path(ARTIFACT_ROOT).expanduser().resolve()
    if ARTIFACT_ROOT is not None
    else repo_root / 'artifacts/brainscales2-toy' / run_label
)
artifact_root.mkdir(parents=True, exist_ok=True)
checkpoint_dir = (
    Path(CHECKPOINT_DIR).expanduser().resolve()
    if CHECKPOINT_DIR is not None
    else artifact_root / 'checkpoint'
)
print('repository:', repo_root)
print('python:', sys.executable)
print('artifacts:', artifact_root)
print('checkpoints:', checkpoint_dir)


In [ ]:
import inspect
import torch
import hxtorch
import hxtorch.perceptron
import hxtorch.spiking as hxsnn
print('Python:', sys.version)
print('torch:', torch.__version__)
print('hxtorch:', getattr(hxtorch, '__version__', 'unknown'))
print('Perceptron Linear:', inspect.signature(hxtorch.perceptron.nn.Linear))
print('Experiment:', inspect.signature(hxsnn.Experiment))
print('LIF:', inspect.signature(hxsnn.LIF))


## One-pass gated pipeline

Set the stage flags in the configuration cell and choose **Run All** once. CPU training and conversion finish before the shared hardware client is requested. The Hagen probe selects the hidden shift, then unlabeled M=1 events select one common deadline margin for both placements. A failed enabled stage records pipeline_status.json, raises, and prevents every later stage from being invoked. Formal hardware stages additionally require the same run's paired zero/selected-margin smoke gate to pass.


In [ ]:
pipeline_status = {}
status_path = artifact_root / 'pipeline_status.json'

def write_pipeline_status():
    status_path.write_text(
        json.dumps(pipeline_status, indent=2, sort_keys=True, default=str),
        encoding='utf-8',
    )

def run_stage(name, action):
    print(f'\n=== {name} ===', flush=True)
    started = time.monotonic()
    pipeline_status[name] = {'status': 'running'}
    write_pipeline_status()
    try:
        details = action()
    except Exception as error:
        pipeline_status[name] = {
            'status': 'failed',
            'error_type': type(error).__name__,
            'error': str(error),
            'elapsed_s': time.monotonic() - started,
        }
        write_pipeline_status()
        raise
    pipeline_status[name] = {
        'status': 'passed',
        'elapsed_s': time.monotonic() - started,
        'details': details if isinstance(details, dict) else {},
    }
    write_pipeline_status()
    return details

def run_cli(*arguments):
    command = [sys.executable, str(CLI), *(str(value) for value in arguments)]
    print(' '.join(command), flush=True)
    phase_index = arguments.index('--phase') + 1
    phase = str(arguments[phase_index])
    timeout_s = CLI_TIMEOUTS_S[phase]
    process = subprocess.Popen(command, cwd=repo_root, start_new_session=True)
    try:
        returncode = process.wait(timeout=timeout_s)
    except subprocess.TimeoutExpired:
        os.killpg(process.pid, signal.SIGKILL)
        process.wait()
        raise TimeoutError(f'{phase} exceeded {timeout_s} seconds')
    if returncode != 0:
        raise subprocess.CalledProcessError(returncode, command)

def train_and_convert():
    run_cli(
        '--phase', 'train', '--task', 'yinyang', '--architecture', 'yy-30', '--activation', TOY_ACTIVATION,
        '--output-dir', checkpoint_dir,
    )
    run_cli(
        '--phase', 'convert', '--task', 'yinyang', '--architecture', 'yy-30', '--activation', TOY_ACTIVATION,
        '--checkpoint', checkpoint_dir / 'checkpoint.pt',
        '--output-dir', checkpoint_dir,
    )
    return {'checkpoint_dir': checkpoint_dir}

def require_checkpoints():
    required = [checkpoint_dir / 'checkpoint.pt', checkpoint_dir / 'converted_checkpoint.pt']
    missing = [path for path in required if not path.is_file()]
    if missing:
        raise FileNotFoundError(
            f'Missing checkpoints: {missing}. Enable RUN_TRAIN or set CHECKPOINT_DIR.'
        )
    return {'checkpoint': required[0], 'converted_checkpoint': required[1]}

if RUN_TRAIN:
    run_stage('train-convert', train_and_convert)

downstream_requested = any((
    RUN_LOCAL_REPLAY,
    RUN_HAGEN_PROBE,
    RUN_MARGIN_CALIBRATION,
    RUN_HARDWARE_SMOKE,
    RUN_YINYANG_FULL,
    RUN_MNIST_BENCHMARK,
))
if downstream_requested:
    run_stage('checkpoint-ready', require_checkpoints)

if RUN_LOCAL_REPLAY:
    run_stage(
        'local-replay',
        lambda: run_cli(
            '--phase', 'local-eval', '--task', 'yinyang', '--architecture', 'yy-30', '--activation', TOY_ACTIVATION,
            '--checkpoint', checkpoint_dir / 'checkpoint.pt',
            '--converted-checkpoint', checkpoint_dir / 'converted_checkpoint.pt',
            '--pwm-backend', 'torch', '--pool-backend', 'replay',
            '--output-dir', artifact_root / 'replay',
        ),
    )

hardware_requested = any((
    RUN_SERVICE_PREFLIGHT,
    RUN_HAGEN_PROBE,
    RUN_MARGIN_CALIBRATION,
    RUN_HARDWARE_SMOKE,
    RUN_YINYANG_FULL,
    RUN_MNIST_BENCHMARK,
))

def configure_hardware():
    global HAGEN_CALIBRATION_PATH, SPIKING_CALIBRATION_PATH
    demos_root = Path('/tmp/brainscales2-demos')
    if not demos_root.is_dir():
        subprocess.run(
            [
                'git', 'clone', '--depth', '1', '--branch',
                'jupyter-notebooks-experimental',
                'https://github.com/electronicvisions/brainscales2-demos.git',
                str(demos_root),
            ],
            check=True,
        )
    if str(demos_root) not in sys.path:
        sys.path.insert(0, str(demos_root))
    from _static.common.helpers import save_nightly_calibration, setup_hardware_client
    for attempt in range(1, HARDWARE_CLIENT_MAX_ATTEMPTS + 1):
        try:
            setup_hardware_client()
            break
        except RuntimeError:
            if attempt == HARDWARE_CLIENT_MAX_ATTEMPTS:
                raise
            delay_s = HARDWARE_CLIENT_RETRY_BACKOFF_S * attempt
            print(
                f'hardware client attempt {attempt} failed; retrying in {delay_s:.0f} s',
                flush=True,
            )
            time.sleep(delay_s)

    calibration_dir = artifact_root / 'calibration'
    calibration_dir.mkdir(parents=True, exist_ok=True)
    if HAGEN_CALIBRATION_PATH is None:
        HAGEN_CALIBRATION_PATH = calibration_dir / 'hagen_cocolist.pbin'
        if not HAGEN_CALIBRATION_PATH.is_file():
            save_nightly_calibration(HAGEN_CALIBRATION_PATH.name, folder=str(calibration_dir))
    else:
        HAGEN_CALIBRATION_PATH = Path(HAGEN_CALIBRATION_PATH).expanduser().resolve()
    if SPIKING_CALIBRATION_PATH is None:
        SPIKING_CALIBRATION_PATH = calibration_dir / 'spiking_cocolist.pbin'
        if not SPIKING_CALIBRATION_PATH.is_file():
            save_nightly_calibration(SPIKING_CALIBRATION_PATH.name, folder=str(calibration_dir))
    else:
        SPIKING_CALIBRATION_PATH = Path(SPIKING_CALIBRATION_PATH).expanduser().resolve()

    missing = [
        path for path in (HAGEN_CALIBRATION_PATH, SPIKING_CALIBRATION_PATH)
        if not path.is_file()
    ]
    if missing:
        raise FileNotFoundError(f'Missing calibration files: {missing}')
    return {
        'hagen_calibration': HAGEN_CALIBRATION_PATH,
        'spiking_calibration': SPIKING_CALIBRATION_PATH,
    }

if hardware_requested:
    run_stage('hardware-client-calibration', configure_hardware)

def run_hagen_preflight():
    code = f'''
import hxtorch
print("initializing explicit Hagen calibration", flush=True)
try:
    hxtorch.init_hardware(hxtorch.CalibrationPath({str(HAGEN_CALIBRATION_PATH)!r}))
    print("Hagen initialization passed", flush=True)
    print("chip:", hxtorch.get_unique_identifier(), flush=True)
finally:
    try:
        hxtorch.release_hardware()
        print("hardware released", flush=True)
    except Exception as error:
        print("release failed:", type(error).__name__, error, flush=True)
'''
    process = subprocess.Popen(
        [sys.executable, '-c', code], cwd=repo_root,
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True,
        start_new_session=True,
    )
    timed_out = False
    try:
        stdout, stderr = process.communicate(timeout=PREFLIGHT_TIMEOUT_S)
    except subprocess.TimeoutExpired:
        timed_out = True
        os.killpg(process.pid, signal.SIGKILL)
        stdout, stderr = process.communicate()
    result = {
        'returncode': process.returncode, 'timed_out': timed_out,
        'stdout': stdout, 'stderr': stderr,
    }
    (artifact_root / 'service_preflight.json').write_text(
        json.dumps(result, indent=2), encoding='utf-8'
    )
    print(stdout)
    if stderr:
        print(stderr, file=sys.stderr)
    if timed_out:
        raise TimeoutError(f'Hagen initialization exceeded {PREFLIGHT_TIMEOUT_S} seconds')
    if process.returncode != 0:
        raise RuntimeError(f'Hagen initialization failed with exit code {process.returncode}')
    return result

if RUN_SERVICE_PREFLIGHT:
    run_stage('hagen-service-preflight', run_hagen_preflight)

def calibration_args():
    return [
        '--hagen-calibration', HAGEN_CALIBRATION_PATH,
        '--spiking-calibration', SPIKING_CALIBRATION_PATH,
    ]

def base_hardware_args():
    weight_args = (
        [] if NEURON_WEIGHT_CALIBRATION_PATH is None else
        ['--neuron-weight-calibration', Path(NEURON_WEIGHT_CALIBRATION_PATH).expanduser().resolve()]
    )
    return [
        *weight_args,
        '--activation', TOY_ACTIVATION,
        '--checkpoint', checkpoint_dir / 'checkpoint.pt',
        '--converted-checkpoint', checkpoint_dir / 'converted_checkpoint.pt',
        '--pwm-backend', 'hagen-hardware',
        '--pool-backend', 'hardware',
        '--pooling-domain', POOLING_DOMAIN,
        '--temporal-pool-estimator', TEMPORAL_POOL_ESTIMATOR,
        '--deadline-s', BASE_DEADLINE_S,
        '--hagen-hidden-shift', HAGEN_HIDDEN_SHIFT,
        '--relu-boundary', RELU_BOUNDARY,
        '--threshold', SPIKING_THRESHOLD,
        '--input-fan-in', SPIKING_INPUT_FAN_IN,
        '--pool-sample-chunk-size', POOL_SAMPLE_CHUNK_SIZE,
        '--pool-replica-sample-budget', POOL_REPLICA_SAMPLE_BUDGET,
        '--pool-calibration-trial-chunk-size', POOL_CALIBRATION_TRIAL_CHUNK_SIZE,
        '--hagen-row-chunk-size', HAGEN_ROW_CHUNK_SIZE,
        '--condition-worker-max-attempts', CONDITION_WORKER_MAX_ATTEMPTS,
        '--condition-worker-retry-backoff-s', CONDITION_WORKER_RETRY_BACKOFF_S,
        '--condition-worker-idle-timeout-s', CONDITION_WORKER_IDLE_TIMEOUT_S,
        *calibration_args(),
    ]

def deadline_margin_args():
    if DEADLINE_MARGIN_PATH is None or not Path(DEADLINE_MARGIN_PATH).is_file():
        raise RuntimeError('Run margin calibration or set DEADLINE_MARGIN_PATH')
    return ['--deadline-margin-json', DEADLINE_MARGIN_PATH, '--include-zero-margin-control']

def common_hardware_args():
    return [*base_hardware_args(), *deadline_margin_args()]

def probe_hagen():
    output_dir = artifact_root / 'hagen_probe'
    run_cli(
        '--phase', 'probe-hagen', '--task', 'yinyang', '--architecture', 'yy-30', '--activation', TOY_ACTIVATION,
        '--checkpoint', checkpoint_dir / 'checkpoint.pt',
        '--converted-checkpoint', checkpoint_dir / 'converted_checkpoint.pt',
        '--pwm-backend', 'hagen-hardware',
        '--hagen-hidden-shift', HAGEN_HIDDEN_SHIFT,
        '--relu-boundary', RELU_BOUNDARY,
        '--output-dir', output_dir,
        *calibration_args(),
    )
    payload = json.loads((output_dir / 'hagen_probe.json').read_text(encoding='utf-8'))
    selected = payload['hidden_shift_calibration']['selected']
    shift = int(selected['shift'])
    if shift < 0 or shift > 7:
        raise RuntimeError(f'Invalid recommended Hagen hidden shift: {shift}')
    return {
        'selected_shift': shift,
        'normalized_mse': selected['normalized_mse'],
        'saturation_rate': selected['saturation_rate'],
    }

if RUN_HAGEN_PROBE:
    probe_result = run_stage('hagen-probe', probe_hagen)
    HAGEN_HIDDEN_SHIFT = int(probe_result['selected_shift'])
    print('Using probe-selected HAGEN_HIDDEN_SHIFT =', HAGEN_HIDDEN_SHIFT)

def resolve_deadline_margin_path():
    global DEADLINE_MARGIN_PATH
    if DEADLINE_MARGIN_PATH is None:
        DEADLINE_MARGIN_PATH = artifact_root / 'deadline_margin' / 'deadline_margin.json'
    DEADLINE_MARGIN_PATH = Path(DEADLINE_MARGIN_PATH).expanduser().resolve()
    if not DEADLINE_MARGIN_PATH.is_file():
        raise FileNotFoundError(DEADLINE_MARGIN_PATH)
    payload = json.loads(DEADLINE_MARGIN_PATH.read_text(encoding='utf-8'))
    if not payload.get('viable') or payload.get('selected_margin_s') is None:
        raise RuntimeError(f'No viable deadline margin: {payload.get("structural_floor")}')
    return {
        'path': DEADLINE_MARGIN_PATH,
        'selected_margin_s': payload['selected_margin_s'],
        'selected_deadline_s': payload['selected_deadline_s'],
    }

def calibrate_deadline_margin():
    output_dir = artifact_root / 'deadline_margin'
    run_cli(
        '--phase', 'calibrate-margin', '--task', 'yinyang', '--architecture', 'yy-30',
        '--margin-calibration-samples', MARGIN_CALIBRATION_SAMPLES,
        '--margin-calibration-trials', MARGIN_CALIBRATION_TRIALS,
        '--margin-diagnostic-deadline-s', MARGIN_DIAGNOSTIC_DEADLINE_S,
        '--margin-max-s', MARGIN_MAX_S, '--margin-step-s', MARGIN_STEP_S,
        '--margin-target-sample-miss-rate', MARGIN_TARGET_SAMPLE_MISS_RATE,
        '--margin-confidence', MARGIN_CONFIDENCE,
        '--margin-bootstrap-iterations', MARGIN_BOOTSTRAP_ITERATIONS,
        '--output-dir', output_dir, *base_hardware_args(),
    )
    return resolve_deadline_margin_path()

if RUN_MARGIN_CALIBRATION:
    run_stage('deadline-margin-calibration', calibrate_deadline_margin)
elif RUN_HARDWARE_SMOKE or RUN_YINYANG_FULL or RUN_MNIST_BENCHMARK:
    run_stage('deadline-margin-ready', resolve_deadline_margin_path)

def validate_smoke(output_dir):
    metrics_path = output_dir / 'metrics.csv'
    required_artifacts = [
        metrics_path,
        output_dir / 'activation_error_by_code.csv',
        output_dir / 'deadline_margin_comparison.csv',
        output_dir / 'manifest.json',
        output_dir / 'predictions.csv',
        output_dir / 'intermediates.pt',
    ]
    missing = [path for path in required_artifacts if not path.is_file()]
    if missing:
        raise FileNotFoundError(f'Smoke did not produce required artifacts: {missing}')
    with metrics_path.open(newline='', encoding='utf-8') as handle:
        rows = [row for row in csv.DictReader(handle) if row.get('pool_size')]
    observed_sizes = {int(row['pool_size']) for row in rows}
    if not {1, 4}.issubset(observed_sizes):
        raise RuntimeError(f'Smoke is missing M=1 or M=4 metrics: {observed_sizes}')
    margin_payload = json.loads(Path(DEADLINE_MARGIN_PATH).read_text(encoding='utf-8'))
    expected_margins = {0.0, float(margin_payload['selected_margin_s'])}
    observed_margins = {float(row['deadline_margin_s']) for row in rows}
    if observed_margins != expected_margins:
        raise RuntimeError(f'Smoke deadline variants disagree: {observed_margins}')
    for row in rows:
        if row['pooling_domain'] != POOLING_DOMAIN:
            raise RuntimeError(f'Unexpected smoke pooling domain: {row}')
        if POOLING_DOMAIN == 'potential':
            if int(row['hagen_avg']) != int(row['pool_size']) or int(row['temporal_pool_size']) != 1:
                raise RuntimeError(f'Smoke did not use Hagen avg=M with one LIF: {row}')
        miss_rate = float(row['neuron_miss_rate'])
        multi_rate = float(row['multi_spike_rate'])
        accuracy = float(row['accuracy'])
        if not math.isfinite(accuracy):
            raise RuntimeError(f'Non-finite smoke accuracy: {row}')
        if miss_rate >= 1.0:
            raise RuntimeError(f'All physical neurons missed in smoke condition: {row}')
        if multi_rate > SMOKE_MAX_MULTI_SPIKE_RATE:
            raise RuntimeError(f'Smoke multi-spike rate exceeds gate: {row}')
    return {
        'conditions': len(rows),
        'pool_sizes': sorted(observed_sizes),
        'maximum_neuron_miss_rate': max(float(row['neuron_miss_rate']) for row in rows),
        'maximum_multi_spike_rate': max(float(row['multi_spike_rate']) for row in rows),
    }

def run_smoke():
    output_dir = artifact_root / 'hardware_smoke'
    run_cli(
        '--phase', 'hardware-smoke', '--task', 'yinyang', '--architecture', 'yy-30', '--activation', TOY_ACTIVATION,
        '--quick', '--output-dir', output_dir,
        *common_hardware_args(),
    )
    return validate_smoke(output_dir)

if RUN_HARDWARE_SMOKE:
    run_stage('hardware-smoke', run_smoke)

formal_requested = RUN_YINYANG_FULL or RUN_MNIST_BENCHMARK
if formal_requested and not RUN_HARDWARE_SMOKE:
    pipeline_status['formal-experiment'] = {
        'status': 'blocked',
        'error': 'Enable RUN_HARDWARE_SMOKE; formal stages require a passing same-run smoke gate.',
    }
    write_pipeline_status()
    raise RuntimeError(pipeline_status['formal-experiment']['error'])

if RUN_YINYANG_FULL:
    run_stage(
        'yinyang-full',
        lambda: run_cli(
            '--phase', 'hardware-eval', '--task', 'yinyang', '--architecture', 'yy-30', '--activation', TOY_ACTIVATION,
            '--pool-sizes', 1, 2, 4, 8, 16,
            '--output-dir', artifact_root / 'yinyang_full',
            *common_hardware_args(),
        ),
    )

if RUN_MNIST_BENCHMARK:
    mnist_checkpoint = artifact_root / 'mnist_checkpoint'
    required = [mnist_checkpoint / 'checkpoint.pt', mnist_checkpoint / 'converted_checkpoint.pt']
    missing = [path for path in required if not path.is_file()]
    if missing:
        raise FileNotFoundError(f'Missing MNIST checkpoints: {missing}')
    run_stage(
        'mnist-benchmark',
        lambda: run_cli(
            '--phase', 'hardware-eval', '--task', 'mnist', '--architecture', 'mnist-30', '--activation', TOY_ACTIVATION,
            '--checkpoint', required[0], '--converted-checkpoint', required[1],
            '--pwm-backend', 'hagen-hardware', '--pool-backend', 'hardware',
            '--hagen-hidden-shift', HAGEN_HIDDEN_SHIFT,
            '--relu-boundary', RELU_BOUNDARY,
            '--threshold', SPIKING_THRESHOLD,
            '--input-fan-in', SPIKING_INPUT_FAN_IN,
            '--pool-sample-chunk-size', POOL_SAMPLE_CHUNK_SIZE,
            '--pool-replica-sample-budget', POOL_REPLICA_SAMPLE_BUDGET,
            '--pool-calibration-trial-chunk-size', POOL_CALIBRATION_TRIAL_CHUNK_SIZE,
            '--hagen-row-chunk-size', HAGEN_ROW_CHUNK_SIZE,
            '--condition-worker-max-attempts', CONDITION_WORKER_MAX_ATTEMPTS,
            '--condition-worker-retry-backoff-s', CONDITION_WORKER_RETRY_BACKOFF_S,
            '--condition-worker-idle-timeout-s', CONDITION_WORKER_IDLE_TIMEOUT_S,
            '--max-test-samples', MNIST_HARDWARE_SAMPLES,
            '--output-dir', artifact_root / 'mnist_benchmark',
            *calibration_args(),
        ),
    )

print('Pipeline status:', json.dumps(pipeline_status, indent=2, default=str))


## Result summary

The pipeline keeps experiment logic in the CLI and displays only the manifests and runtime estimates written by completed stages.


In [ ]:
for manifest_path in sorted(artifact_root.rglob('manifest.json')):
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    print(
        manifest_path.relative_to(artifact_root),
        {
            'task': manifest.get('task'),
            'test_samples': manifest.get('test_samples'),
            'conditions': len(manifest.get('conditions', [])),
        },
    )
for runtime_path in sorted(artifact_root.rglob('runtime.json')):
    runtime = json.loads(runtime_path.read_text(encoding='utf-8'))
    print(runtime_path.relative_to(artifact_root), runtime.get('mnist_estimates'))

analysis_fields = [
    'pool_size', 'hagen_avg', 'temporal_pool_size', 'placement', 'accuracy',
    'base_deadline_s', 'deadline_margin_s', 'resolved_deadline_s',
    'neuron_miss_rate_nominal_zero', 'neuron_miss_rate_nominal_positive',
    'all_miss_rate_nominal_zero', 'all_miss_rate_nominal_positive',
    'nonmiss_activation_mae_uint5', 'nonmiss_activation_bias_uint5',
    'oracle_miss_repair_accuracy', 'torch_readout_accuracy',
    'torch_oracle_miss_repair_accuracy',
]
for metrics_path in sorted(artifact_root.rglob('metrics.csv')):
    with metrics_path.open(newline='', encoding='utf-8') as handle:
        rows = [row for row in csv.DictReader(handle) if row.get('pool_size')]
    print(metrics_path.relative_to(artifact_root))
    for row in rows:
        print({field: row.get(field) for field in analysis_fields})
for comparison_path in sorted(artifact_root.rglob('deadline_margin_comparison.csv')):
    with comparison_path.open(newline='', encoding='utf-8') as handle:
        comparisons = list(csv.DictReader(handle))
    print(comparison_path.relative_to(artifact_root), comparisons)
for code_path in sorted(artifact_root.rglob('activation_error_by_code.csv')):
    with code_path.open(newline='', encoding='utf-8') as handle:
        observed = [row for row in csv.DictReader(handle) if int(row['pool_observations']) > 0]
    print(code_path.relative_to(artifact_root), 'observed code rows =', len(observed))
